## Attempt

I attempted to work with PCA to reduce dimensionality with my created features post-aggregation. These features are the general summary statistics for amount and include some date-related measures. After standard scaling variables, I implemented an XGBoost model with a base score of 0.5 and log loss function.

In addition, I created a function that compares the correlation between these variables and FPF_TARGET; I also calculate the mean of these variables by FPF_TARGET label.

## Main Findings

- PCA may not exactly be the right for this model; it reduced my baseline model accuracy down to 50%. This can be due to limited data, however. More testing will need to be done to determine whether or not I will continue to work with PCA.

- mean_pos_amount, range_date_posted, first_date_posted, max_amount, fin_balance, mean_date_posted have the highest correlation to FPF_TARGET. This can indicate that these variables are quite helpful in differentiating between the FPF_TARGET labels.

- There are noticeable differences in variables means between the two FPF_TARGET labels for the following variables: total_amount, fin_balance, total_pos_amount, mean_neg_amount, mean_amount, min_amount, max_amount, range_date_posted, date_mean_median_diff_posted. Again, this can indicate that these variables are quite helpful in differentiating between the FPF_TARGET labels.

- This baseline model achieves an accuracy on the sample data of ~60%--slightly better than guessing. My next steps are with optimization to improve the model to hopefully achieve the baseline accuracy of 90%....

In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from abc import ABC, abstractmethod
from pathlib import Path

from cashflow import ScorableModelTemplate
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


import warnings
warnings.filterwarnings("ignore")

In [6]:
class ScorableModel(ScorableModelTemplate):

    def predict(self, raw_consumer_file: str, raw_transactions_file: str):
        """Input argument will vary. See you competition's template.

        :param raw_files: list of file path strings, depends on competition
        :return predictions: dataframe or np.array, depends on competition
        """
        # Implement this: may return random predictions for the first assignment
        df_consumers = self.process_inputs(raw_consumer_file, raw_transactions_file) #load processed features
        
        def helper(group_df):
            result = {
                'num_transactions': group_df['amount'].count(),
                'total_amount': group_df['amount'].sum(),
                'fin_balance': group_df['total_balance'].mean(),
                'total_pos_amount': group_df[group_df['amount'] > 0]['amount'].sum(),
                'total_neg_amount': group_df[group_df['amount'] < 0]['amount'].sum(),
                'mean_pos_amount': group_df[group_df['amount'] > 0]['amount'].mean(),
                'mean_neg_amount': group_df[group_df['amount'] < 0]['amount'].mean(),
                'mean_amount': group_df['amount'].mean(),
                'amount_mean_median_diff': group_df['amount'].mean() - group_df['amount'].median(),
                'min_amount': group_df['amount'].min(),
                'max_amount': group_df['amount'].max(),
                'common_category': group_df['category'].mode()[0],
                'date': group_df['evaluation_date'].min().timestamp(),
                'first_date_posted': group_df['posted_date'].min().timestamp(),
                'last_date_posted': group_df['posted_date'].max().timestamp(),
                'range_date_posted': (group_df['posted_date'].max() - group_df['posted_date'].min()).days,
                'mode_day_posted': group_df['posted_day'].mode()[0],
                'mean_date_posted': group_df['posted_date'].mean().timestamp(),
                'date_mean_median_diff_posted': (group_df['posted_date'].mean() - group_df['posted_date'].median()).days,
                'FPF_TARGET': group_df['FPF_TARGET'].mean()
            }
            if len(group_df[group_df['amount'] > 0]) == 0:
                result['total_pos_amount'] = 0
                result['mean_pos_amount'] = 0
            elif len(group_df[group_df['amount'] < 0]) == 0:
                result['total_neg_amount'] = 0
                result['mean_neg_amount'] = 0
            return pd.Series(result)
        
        # === Step 3: Aggregate ===

        ##################################TRAINING DATA####################################
        df_train = self.process_inputs("~/private/DSC 190/mlc/consumers_test_data.parquet", "~/private/DSC 190/mlc/transactions_test_data.parquet") #load processed features
        
        aggregated_train_data = df_train.groupby('masked_consumer_id').apply(helper).reset_index()
        aggregated_train_encoded = pd.get_dummies(aggregated_train_data.drop(columns=['masked_consumer_id']), drop_first=True)

        # Separate target
        train_target = aggregated_train_encoded['FPF_TARGET']
        train_features = aggregated_train_encoded.drop(columns='FPF_TARGET')

        # Standardize
        scaler = StandardScaler()
        train_features_scaled = scaler.fit_transform(train_features)
        
        # === Step 5: Run PCA ===
        pca = PCA()
        train_pca_components = pca.fit_transform(train_features_scaled)
        cumulative_var = np.cumsum(pca.explained_variance_ratio_)
        n_components_95 = np.argmax(cumulative_var >= 0.95) + 1
        
        # Create DataFrame with only the most significant components
        pca_train_df = pd.DataFrame(
            train_pca_components[:, :n_components_95],
            columns=[f'PC{i+1}' for i in range(n_components_95)]
        )
        ##################################TESTING DATA####################################
        aggregated_data = df_consumers.groupby('masked_consumer_id').apply(helper).reset_index()
        aggregated_encoded = pd.get_dummies(aggregated_data.drop(columns=['masked_consumer_id']), drop_first=True)
        
        target = aggregated_encoded['FPF_TARGET']
        features = aggregated_encoded.drop(columns='FPF_TARGET')

        features_scaled = scaler.transform(features)
        
        pca_components = pca.transform(features_scaled)
        pca_df = pd.DataFrame(
            pca_components[:, :n_components_95],
            columns=[f'PC{i+1}' for i in range(n_components_95)]
        )
        
        
        # Step 3: Initialize and train the XGBoost classifier
        model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss',base_score=0.5)
        model.fit(pca_train_df, train_target)

        # Step 4: Make predictions
        y_pred = model.predict(pca_df)
        print(f"Accuracy Test: {accuracy_score(target, y_pred):.2f}")

        return y_pred

    def process_inputs(self, raw_consumer_file: str, raw_transactions_file: str):
        """Input argument will vary. See you competition's template.

        :param raw_files: list of file path strings, depends on competition
        :return: anything needed for you model to make predictions, e.g. features or processed data
        """
        # Implement this: only need to read in files for first assignment
        df_consumer = pd.read_parquet(raw_consumer_file)
        df_transactions = pd.read_parquet(raw_transactions_file)
        merged_df = df_transactions.merge(df_consumer, on = "masked_consumer_id")
        filtered_df = merged_df[merged_df["posted_date"]< merged_df["evaluation_date"]]
        
        filtered_df['evaluation_date'] = pd.to_datetime(filtered_df['evaluation_date'])
        filtered_df['evaluation_day'] = filtered_df['evaluation_date'].dt.dayofweek

        filtered_df['posted_date'] = pd.to_datetime(filtered_df['posted_date'])
        filtered_df['posted_day'] = filtered_df['posted_date'].dt.dayofweek
        
        return filtered_df

# Intialize, runs: __check_rep__ to validate class
model = ScorableModel() # error will be raised if the above is not implemented correctly

Accuracy Test: 0.00


In [7]:
_, true_output = model.load_test_case()
predictions = model.predict("~/private/DSC 190/mlc/consumers_test_data.parquet", "~/private/DSC 190/mlc/transactions_test_data.parquet")

Accuracy Test: 1.00


In [18]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# === Step 1: Load and preprocess data ===
my_df = model.process_inputs("~/private/DSC 190/mlc/consumers_test_data.parquet", 
                             "~/private/DSC 190/mlc/transactions_test_data.parquet")

my_df['evaluation_date'] = pd.to_datetime(my_df['evaluation_date'])
my_df['evaluation_day'] = my_df['evaluation_date'].dt.dayofweek

my_df['posted_date'] = pd.to_datetime(my_df['posted_date'])
my_df['posted_day'] = my_df['posted_date'].dt.dayofweek

# === Step 2: Define aggregation helper ===
def helper(group_df):
    result = {
        'num_transactions': group_df['amount'].count(),
        'total_amount': group_df['amount'].sum(),
        'fin_balance': group_df['total_balance'].mean(),
        'total_pos_amount': group_df[group_df['amount'] > 0]['amount'].sum(),
        'total_neg_amount': group_df[group_df['amount'] < 0]['amount'].sum(),
        'mean_pos_amount': group_df[group_df['amount'] > 0]['amount'].mean(),
        'mean_neg_amount': group_df[group_df['amount'] < 0]['amount'].mean(),
        'mean_amount': group_df['amount'].mean(),
        'amount_mean_median_diff': group_df['amount'].mean() - group_df['amount'].median(),
        'min_amount': group_df['amount'].min(),
        'max_amount': group_df['amount'].max(),
        'common_category': group_df['category'].mode()[0],
        'date': group_df['evaluation_date'].min().timestamp(),
        'first_date_posted': group_df['posted_date'].min().timestamp(),
        'last_date_posted': group_df['posted_date'].max().timestamp(),
        'range_date_posted': (group_df['posted_date'].max() - group_df['posted_date'].min()).days,
        'mode_day_posted': group_df['posted_day'].mode()[0],
        'mean_date_posted': group_df['posted_date'].mean().timestamp(),
        'date_mean_median_diff_posted': (group_df['posted_date'].mean() - group_df['posted_date'].median()).days,
        'FPF_TARGET': group_df['FPF_TARGET'].mean()
    }
    if len(group_df[group_df['amount'] > 0]) == 0:
        result['total_pos_amount'] = 0
        result['mean_pos_amount'] = 0
    elif len(group_df[group_df['amount'] < 0]) == 0:
        result['total_neg_amount'] = 0
        result['mean_neg_amount'] = 0
    return pd.Series(result)

# === Step 3: Aggregate ===
aggregated_data = my_df.groupby('masked_consumer_id').apply(helper).reset_index()

# === Step 4: Encode categorical and scale ===
# Convert 'common_category' to dummy variables (one-hot encoding)
aggregated_encoded = pd.get_dummies(aggregated_data.drop(columns=['masked_consumer_id']), drop_first=True)

# Separate target
target = aggregated_encoded['FPF_TARGET']
features = aggregated_encoded.drop(columns='FPF_TARGET')

# Standardize
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# === Step 5: Run PCA ===
#pca = PCA()
#pca_components = pca.fit_transform(features_scaled)

# Create a DataFrame with principal components
#pca_df = pd.DataFrame(pca_components, columns=[f'PC{i+1}' for i in range(pca_components.shape[1])])
#pca_df['FPF_TARGET'] = target.values

# === Step 6: Correlation function ===
def correlation_with_x(df, x="FPF_TARGET"):
    if x not in df.columns:
        raise ValueError(f"{x} is not a column in the DataFrame")
    numeric_df = df.select_dtypes(include='number')
    correlation_row = numeric_df.corr().loc[x].drop(x)
    return correlation_row.to_frame(name=f'correlation_with_{x}').T

# === Step 7: Run correlation ===
correlation_result = correlation_with_x(aggregated_encoded)
print(correlation_result)

                             num_transactions  total_amount  fin_balance  \
correlation_with_FPF_TARGET          0.037665       0.03886    -0.209119   

                             total_pos_amount  total_neg_amount  \
correlation_with_FPF_TARGET         -0.197923          0.185542   

                             mean_pos_amount  mean_neg_amount  mean_amount  \
correlation_with_FPF_TARGET        -0.288391         0.161204     0.101912   

                             amount_mean_median_diff  min_amount  max_amount  \
correlation_with_FPF_TARGET                 0.015118     0.18759   -0.204316   

                             common_category      date  first_date_posted  \
correlation_with_FPF_TARGET        -0.101483  0.136436           0.288586   

                             last_date_posted  range_date_posted  \
correlation_with_FPF_TARGET          0.166606          -0.294773   

                             mode_day_posted  mean_date_posted  \
correlation_with_FPF_TARGET         

In [17]:
aggregated_data.groupby('FPF_TARGET').describe().xs('mean', level=1, axis=1)

,num_transactions,total_amount,fin_balance,total_pos_amount,total_neg_amount,mean_pos_amount,mean_neg_amount,mean_amount,amount_mean_median_diff,min_amount,max_amount,common_category,date,first_date_posted,last_date_posted,range_date_posted,mode_day_posted,mean_date_posted,date_mean_median_diff_posted
FPF_TARGET,,,,,,,,,,,,,,,,,,,
0.0,391.3125,-1678.809792,8499.759583,74365.074167,-76043.883958,1346.242779,-474.138105,-19.654683,13.921254,-14159.412083,17366.687917,10.583333,1.668067e+09,1.648391e+09,1.667066e+09,216.145833,1.291667,1.658042e+09,-1.229167
1.0,423.2200,48.300600,1097.347400,25993.001200,-25944.700600,346.137383,-100.251264,6.579983,17.784683,-1883.935800,3371.235800,9.100000,1.671294e+09,1.659906e+09,1.671113e+09,129.700000,1.920000,1.665543e+09,0.580000
